使用 “朴素贝叶斯 MultinomialNB” 完成经典的 “垃圾短信识别SMSSpamCollection”

from sklearn.naive_bayes import MultinomialNB


类别
- ham：正常短信
- spam：垃圾短信


In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)

In [16]:
# 1. 加载用户分群 CSV 数据集
df = pd.read_csv("../../../data/ml/SMSSpamCollection", sep="\t", names=["label", "message"])
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
# 2、提取两列数据
#   ├── message：短信文本
#   └── label：短信类别（ham / spam）
X = df["message"]
y = df["label"]

In [18]:
# 3、检查并清理异常数据
# ...

In [19]:
# 4、划分训练集和测试集
#   ├── 训练集：用于学习
#   └── 测试集：用于评估

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(4457,) (1115,) (4457,) (1115,)


In [20]:
# 5、构建 Pipeline
#   ├── TfidfVectorizer
#   │     └── 将短信文本转换为数值特征
#   └── MultinomialNB
#         └── 学习特征与类别之间的关系

model = Pipeline([
  ("tfidf", TfidfVectorizer()),
  ("nb", MultinomialNB())
])

# 使用训练集训练模型
model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('nb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U4](2,)","['ham','spam']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


In [21]:
# 6、使用测试集预测短信类别
# 计算 Accuracy、Precision、Recall、F1 并查看混淆矩阵
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))


              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       959
        spam       1.00      0.72      0.84       156

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115

[[959   0]
 [ 43 113]]
0.9614349775784753


In [22]:
# 7. 预测一条新短信
new_message = [
    "Congratulations! You have won a free prize. Call now!"
]
print("预测结果：", model.predict(new_message))
print("类别概率：", model.predict_proba(new_message))

预测结果： ['spam']
类别概率： [[0.23279655 0.76720345]]


In [23]:
test2_X = [
    "free prize call now",
    "free money claim prize",
    "hello how are you",
    "are we meeting today",
    "where are from?",
    "This is the class and function reference of scikit-learn.",
]

test2_y = [
    "spam",
    "spam",
    "ham",
    "ham",
    "ham",
    "ham",
]

print("预测结果：", model.predict(test2_X))
print("预测结果：", model.predict(test2_X) == test2_y)
print("类别概率：", model.predict_proba(test2_X))

预测结果： ['spam' 'spam' 'ham' 'ham' 'ham' 'ham']
预测结果： [ True  True  True  True  True  True]
类别概率： [[0.26333116 0.73666884]
 [0.20506544 0.79493456]
 [0.99388154 0.00611846]
 [0.99320716 0.00679284]
 [0.9810662  0.0189338 ]
 [0.95034306 0.04965694]]
